# gap_bench — Colab 실험 노트북

**전제:** Drive의 `MyDrive/gap_bench/`에 이 구조로 업로드: 코드는 `src/`, 노트북은 루트.  
**런타임:** CPU (GPU 불필요 — 선택하면 낭비), 고용량 RAM 권장.  
**재개:** 세션 끊기면 [1][2] 후 [5]만 다시 실행 (완료 슬라이스 자동 skip).


## [1] Drive 마운트


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/gap_bench
import sys, os
assert os.path.exists('src/formulation.py'), '폴더 구조 확인'
sys.path.insert(0, 'src')
!ls


## [2] 의존성 (세션마다)


In [ ]:
!pip -q install pulp highspy pyarrow pandas


## [3] 환경 검증 — **전 항목 ✔ 확인 후 진행**


In [ ]:
!python src/toy_test.py


## [4] 트레이스 준비 (최초 1회, ~15–20분)


In [ ]:
import os
if not os.path.exists('traces/conv.parquet'):
    !python src/prep_trace.py
    !rm -f traces/azure_llm_2024_*.csv
else:
    print('parquet 이미 존재 — 건너뜀')
!ls -la traces/


## [5] 실행 파라미터 + 실행 (재개 가능)


In [ ]:
PILOT = True        # False = 본그리드 50슬라이스
TIME_LIMIT = 900    # Colab 세션 제한 감안 900 이하 권장
import run_slices
run_slices.main(pilot=PILOT, time_limit=TIME_LIMIT)


## [6] 점검 (warm_applied / milp_gap / 소요시간)


In [ ]:
import pandas as pd
df = pd.read_csv('results/results.csv')
df['fcfs_gap_lo'] = (df.fcfs_lat - df.milp_incumbent) / df.milp_incumbent
df['fcfs_gap_hi'] = (df.fcfs_lat - df.milp_bound) / df.milp_bound
df['sjf_gap_lo']  = (df.sjf_lat  - df.milp_incumbent) / df.milp_incumbent
df['sjf_gap_hi']  = (df.sjf_lat  - df.milp_bound) / df.milp_bound
ok = bool(df.warm_applied.astype(bool).all())
print('1) warm_applied 전부 True? ->', ok, '' if ok else '*** 문제: DEVLOG 참조 ***')
print(f'2) milp_gap: mean={df.milp_gap.mean():.1%}  max={df.milp_gap.max():.1%}  (목표: max 10% 이하)')
print(f'3) 슬라이스당: mean={df.solve_s.mean()/60:.1f}분  ->  본그리드 50슬라이스 예상 {df.solve_s.mean()*50/3600:.1f}h')
cols = ['slice_id','fcfs_gap_lo','fcfs_gap_hi','sjf_gap_lo','sjf_gap_hi','milp_gap','milp_status','solve_s']
df[cols]


## [7] 본실험 전환: [5]에서 `PILOT = False` 후 재실행

로컬과 같은 Drive 폴더를 쓰므로 results.csv 원장은 공유됩니다 — 동시에 양쪽에서 돌리지는 마세요 (원장 경합).


## [부록] 결과 원장 백업


In [ ]:
import time, shutil, os
os.makedirs('results/backup', exist_ok=True)
dst = f"results/backup/results_{time.strftime('%m%d_%H%M')}.csv"
shutil.copy('results/results.csv', dst)
print('backup ->', dst)
